In [7]:
from roboflow import Roboflow
rf = Roboflow(api_key="038INb0Av0p6eo4CVxxx")
project = rf.workspace("marcuss-workspace").project("utility-poles-kcumt-nt0d2")
version = project.version(2)
dataset = version.download("yolo26")
                      

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Utility-Poles-2 in yolo26:: 100%|██████████| 2899/2899 [00:01<00:00, 1548.71it/s]


In [ ]:
from ultralytics import YOLO
model = YOLO("yolo26m.pt")

In [ ]:
# Train the YOLO model on the downloaded dataset using optimal parameters
results = model.train(
    data="utility-poles-2/data.yaml",
    epochs=100,
    batch=16,
    device=[0,1,2,3],  # Use GPU if available; adjust as needed
    workers=8,
    patience=0,
    optimizer='Adam',  # Adam optimizer is robust/default for most tasks
    lr0=0.001,         # Initial learning rate
    project='utility-pole-training',
    name='yolo26m-utility-poles',
    verbose=True,
    pretrained=True,
    cache=True,
    save_period=10  # Save a checkpoint every 10 epochs for ensembling
)

In [ ]:
# Run inference on a single validation image and display the result

import cv2
from matplotlib import pyplot as plt

# Get a sample image path from the validation set
sample_img_path = "utility-poles-1/valid/images"
import os
sample_images = [f for f in os.listdir(sample_img_path) if f.endswith(".jpg") or f.endswith(".jpeg") or f.endswith(".png")]

if sample_images:
    img_path = os.path.join(sample_img_path, sample_images[0])

    # Run prediction
    results = model(img_path)  # This does inference

    # Visualize the result
    annotated_img = results[0].plot()  # returns an array (BGR by default)

    # Convert from BGR to RGB for matplotlib
    annotated_img_rgb = cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(12, 8))
    plt.imshow(annotated_img_rgb)
    plt.axis("off")
    plt.title("YOLO Inference Result on Sample Image")
    plt.show()
else:
    print("No sample images found in the validation directory.")